In [1]:
import cloudpickle
 
with open("model_SCTMD.pkl", "rb") as f:
    model = cloudpickle.load(f)

In [4]:
from statsmodels.tsa.statespace.structural import UnobservedComponentsResults
 
model = UnobservedComponentsResults.load("model_SCTMD.pkl")
exog_names = ['Current_basis_weight',
 'Starch_uptake__g/m2_',
 'Jet/wire_ratio',
 'Speed_Size_Press',
 'Rod_pressure_Top_Roll',
 'Steam_flow_to_AfterDryers',
 'Moisture_out_of_PreDryer',
 'Linepressure_1st_press_FS__bar_',
 'Linepressure_1st_press_DS__bar_',
 'Consistency_white_water',
 'Conductivity_white_water_B46',
 'pH_measurement_white_water_B41'] 

In [5]:
import pandas as pd
import numpy as np

df = pd.read_parquet(f"data/costimier_continuous.parquet").set_index("Wedge_Time")
df = df[~df.index.duplicated(keep="first")]
df.loc[:,"Starch_uptake__g/m2_"]=df["Starch_uptake_by_paper_Bottom_Roll__g/m2_"]+df["Starch_uptake_by_paper_Top_Roll__g/m2_"]
transition_mask=df['MBS_Current_reel_ID']!=df['MBS_Current_reel_ID'].shift(-1)
itansition=np.where(transition_mask)[0]

temp=df.loc[transition_mask, "MBS_SCT_MD"]
df["MBS_SCT_MD"]=np.nan
df.loc[transition_mask,"MBS_SCT_MD"]=temp
X = df[exog_names]
y = df["MBS_SCT_MD"]

In [6]:
X_test = df[(df.index > "2025-12-1") & (df.index < "2025-12-10")][exog_names]
y_test = df[(df.index > "2025-12-1") & (df.index < "2025-12-10")]["MBS_SCT_MD"]

In [7]:
import seaborn as sns 

sns.set(rc={"figure.figsize":(12, 4)})
sns.set_style('whitegrid')
sns.set_context('notebook')

In [ ]:
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
 
preds = []
lower = []
upper = []
 
refit_every = 50
valid_count = 0
 
endog_refit = np.asarray(model.model.data.orig_endog).reshape(-1)
exog_refit = np.asarray(model.model.data.orig_exog) if model.model.data.orig_exog is not None else None
 
model_cls = model.model.__class__
init_kwargs = model.model._get_init_kwds().copy()
 
for i in tqdm(range(len(y_test)), total=len(y_test), desc="Recursive prediction"):
 
    x_row = X_test.iloc[[i]].to_numpy()
    y_true = y_test.iloc[i]
 
    fcst = model.get_forecast(steps=1, exog=x_row)
    y_hat = fcst.predicted_mean[0]
    ci = fcst.conf_int()
 
    preds.append(y_hat)
    lower.append(ci[0, 0])
    upper.append(ci[0, 1])
 
    if pd.notna(y_true):
        y_row = np.array([y_true])
 
        model = model.append(endog=y_row, exog=x_row, refit=False)
 
        endog_refit = np.concatenate([endog_refit, y_row])
        if exog_refit is not None:
            exog_refit = np.vstack([exog_refit, x_row])
 
        valid_count += 1
 
        if valid_count % refit_every == 0:
            print(f"Full refit at index {i}, timestamp {y_test.index[i]}")
 
            refit_mod = model_cls(
                endog=endog_refit,
                exog=exog_refit,
                **init_kwargs
            )
            model = refit_mod.fit(start_params=model.params, disp=False)
 
pred_df = pd.DataFrame(
    {
        "y_true": y_test.values,
        "y_pred": preds,
        "lower": lower,
        "upper": upper,
    },
    index=y_test.index,
)
 
df_eval = pred_df.dropna(subset=["y_true"])
 
rmse = np.sqrt(mean_squared_error(df_eval["y_true"], df_eval["y_pred"]))
mae = mean_absolute_error(df_eval["y_true"], df_eval["y_pred"])
r2 = r2_score(df_eval["y_true"], df_eval["y_pred"])
 
print("N:", len(df_eval))
print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)
 